# 실습 8 심화 (선택) · LLM으로 설정 · 코드 보안 점검

취약점을 일부러 넣은 가상 설정 파일과 코드를 LLM에게 점검시키고, 정답 목록과 비교합니다.
LLM 점검은 사람 검토와 기존 점검 도구를 보조하는 용도입니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "llm_client.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find project root containing llm_client.py")
sys.path.insert(0, str(PROJECT_ROOT))

from llm_client import create_llm_client

PROVIDER = "mono"
client, MODEL = create_llm_client(provider=PROVIDER, api_key_env="MY_API_KEY")
print(PROVIDER, MODEL)

In [ ]:
from openai.types.chat import ChatCompletionMessageParam

def ask(
    prompt: str,
    system: str | None = None,
    temperature: float = 0.2,
    show: bool = True,
) -> str:
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages: list[ChatCompletionMessageParam] = [
        {"role": "user", "content": prompt}
    ]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    if PROVIDER == "ollama":
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            stream=True,
            max_tokens=500,
        )
    else:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            stream=True,
        )
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [3]:
import json, re

def parse_json(raw):
    """답에서 { ... } 부분만 꺼내 JSON으로 읽음. 없거나 깨졌으면 None"""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

In [4]:
import re

TARGETS = {
    "sshd_config": """1  Port 22
2  PermitRootLogin yes
3  PasswordAuthentication yes
4  PermitEmptyPasswords yes
5  MaxAuthTries 10
6  X11Forwarding yes
7  ClientAliveInterval 0""",
    "app.py": """1  import hashlib, sqlite3, requests
2  DEBUG = True
3  SECRET_KEY = "admin1234"
4  def login(user, pw):
5      h = hashlib.md5(pw.encode()).hexdigest()
6      q = "SELECT * FROM users WHERE id='" + user + "' AND pw='" + h + "'"
7      return sqlite3.connect("bank.db").execute(q).fetchone()
8  def fetch_rate():
9      return requests.get("https://rate.example.com/api", verify=False).json()
10 def log_login(user, pw):
11     print(f"login user={user} pw={pw}")""",
}

ANSWER_KEY = {   # 찾았다고 볼 키워드
    "sshd_config": {"root 로그인": r"root", "비밀번호 인증": r"PasswordAuthentication|비밀번호 인증",
                    "빈 비밀번호": r"PermitEmptyPasswords|빈 (비밀번호|패스워드)", "인증 시도 과다": r"MaxAuthTries|시도",
                    "X11 포워딩": r"X11", "유휴 세션 무제한": r"ClientAlive|유휴|세션"},
    "app.py": {"디버그 모드": r"DEBUG|디버그", "비밀키 하드코딩": r"SECRET_KEY|하드코딩", "MD5": r"md5",
               "SQL 인젝션": r"SQL|인젝션", "인증서 검증 끔": r"verify|인증서", "비밀번호 로그 출력": r"로그.{0,10}(비밀번호|pw)|print"},
}

PROMPT = """[역할] 당신은 금융기관 보안 점검 담당자입니다.
[제약] 아래 내용에 실제로 있는 줄만 근거로 지적한다. 근거 줄 번호를 반드시 적는다.
[출력 형식] JSON으로만 답한다.
{{"findings": [{{"item": "취약점 이름", "severity": "높음/중간/낮음", "line": 줄번호, "fix": "조치"}}]}}
[점검 대상: {name}]
{body}"""

In [5]:
for name, body in TARGETS.items():
    print(f"\n===== {name} =====")
    d = parse_json(ask(PROMPT.format(name=name, body=body), temperature=0.1))
    findings = d.get("findings", []) if d else []
    n_lines = len(body.splitlines())
    for f in findings:
        ok = isinstance(f.get("line"), int) and 1 <= f["line"] <= n_lines        # 근거 줄이 실제로 있는지
        print(f"  [{f.get('severity')}] {f.get('item')} (줄 {f.get('line')}{'' if ok else ' ← 없는 줄'})")
    text = str(findings)
    hit = [k for k, p in ANSWER_KEY[name].items() if re.search(p, text, re.I)]
    print(f"  정답 {len(ANSWER_KEY[name])}개 중 찾음 {len(hit)}개, 놓친 것: {[k for k in ANSWER_KEY[name] if k not in hit] or '없음'}")


===== sshd_config =====
{"findings":[{"item":"root 계정 SSH 직접 로그인 허용","severity":"높음","line":2,"fix":"PermitRootLogin no로 변경하고, 일반 계정 로그인 후 sudo를 사용하도록 설정한다."},{"item":"비밀번호 기반 SSH 인증 허용","severity":"중간","line":3,"fix":"PasswordAuthentication no로 변경하고 공개키 기반 인증을 사용한다."},{"item":"빈 비밀번호 계정의 SSH 로그인 허용","severity":"높음","line":4,"fix":"PermitEmptyPasswords no로 변경하고 빈 비밀번호 계정을 제거한다."},{"item":"SSH 인증 재시도 횟수 과다","severity":"중간","line":5,"fix":"MaxAuthTries 값을 3~5 이하로 낮춘다."},{"item":"X11 포워딩 허용","severity":"낮음","line":6,"fix":"X11Forwarding no로 변경한다."},{"item":"SSH 유휴 세션 타임아웃 비활성화","severity":"중간","line":7,"fix":"ClientAliveInterval을 300 등 적절한 값으로 설정하고 ClientAliveCountMax를 함께 제한한다."}]}
  [높음] root 계정 SSH 직접 로그인 허용 (줄 2)
  [중간] 비밀번호 기반 SSH 인증 허용 (줄 3)
  [높음] 빈 비밀번호 계정의 SSH 로그인 허용 (줄 4)
  [중간] SSH 인증 재시도 횟수 과다 (줄 5)
  [낮음] X11 포워딩 허용 (줄 6)
  [중간] SSH 유휴 세션 타임아웃 비활성화 (줄 7)
  정답 6개 중 찾음 6개, 놓친 것: 없음

===== app.py =====
{"findings":[{"item":"디버그 모드 활성화","severity":"중간","line":2,"fix":"운영 환경에서는 DE

### 관찰 포인트
- 잘 알려진 항목(root 로그인, SQL 인젝션)은 잘 찾지만 덜 알려진 항목(유휴 세션, 로그에 비밀번호)은 놓치기 쉽습니다.
- '없는 줄'로 표시된 지적은 근거가 없는 내용입니다. 줄 번호 확인처럼 기계적으로 검증할 장치를 둡니다.
- 실제 설정 · 코드를 외부 LLM에 보낼 때는 비밀값 · 내부 주소를 먼저 가립니다(실습 2b).